# Sequence Parallelism: Hands-On Tutorial

**A complete demonstration of Tensor Parallelism + Sequence Parallelism vs Vanilla Tensor Parallelism.**

## What You'll Learn

1. **Vanilla TP**: All-reduce after row-parallel layers, LayerNorm/Dropout on full tensors
2. **TP + SP**: Reduce-scatter + all-gather, LayerNorm/Dropout on sharded tensors  
3. **Memory savings**: SP reduces activation memory by 30-50%
4. **Same communication**: All-reduce = reduce-scatter + all-gather

## Key Insight

```
Vanilla TP:
  [matmul partial] -> [ALL-REDUCE] -> [LayerNorm on (B, S, h)] 
                      (fused)         full tensor - WASTED MEMORY!

TP + SP:
  [matmul partial] -> [REDUCE-SCATTER] -> [LayerNorm on (B, S/N, h)] -> [ALL-GATHER]
                      (explicit)          1/N of tensor - MEMORY SAVED!  (explicit)
```

Same bytes transferred, but we do useful work between reduce-scatter and all-gather!

**Requirements:** 2+ NVIDIA GPUs (RunPod, Lambda Labs, etc.)

---

## Part 0: Environment Setup

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Number of GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory / 1e9:.1f} GB)')

## Part 1: Setup Tutorial Files

The tutorial consists of several Python files:
- `config.py`: Shared model configuration
- `vanilla_tp.py`: Vanilla Tensor Parallelism (baseline)
- `tp_sp.py`: Tensor Parallelism + Sequence Parallelism
- `trace_shapes.py`: Visualize activation shapes
- `analyze_comm.py`: Analyze communication patterns

In [ ]:
# Download or copy the tutorial files
# If running on RunPod, upload the sp_tutorial folder or copy these files

# Check if files exist
import os
files = ['config.py', 'vanilla_tp.py', 'tp_sp.py', 'trace_shapes.py', 'analyze_comm.py']
for f in files:
    if os.path.exists(f):
        print(f"[OK] {f}")
    else:
        print(f"[MISSING] {f} - please upload this file")

## Part 2: Model Configuration

We use a medium-sized transformer to clearly see memory differences:

In [ ]:
from config import *

print("Model Configuration:")
print(f"  d_model:    {D_MODEL}")
print(f"  n_heads:    {N_HEADS} (d_head = {D_HEAD})")
print(f"  d_ff:       {D_FF}")
print(f"  n_layers:   {N_LAYERS}")
print(f"  vocab_size: {VOCAB_SIZE}")
print()
print("Benchmark Configuration:")
print(f"  batch_size: {BATCH_SIZE}")
print(f"  seq_len:    {SEQ_LEN}")
print()
n_gpus = torch.cuda.device_count()
print(f"With TP={n_gpus}:")
print(f"  S/N = {SEQ_LEN // n_gpus} tokens per GPU (in SP region)")
print(f"  h/N = {D_MODEL // n_gpus} hidden dim per GPU (in TP region)")

## Part 3: Run Vanilla Tensor Parallelism (Baseline)

In vanilla TP:
- Column-parallel layers (W_q, W_k, W_v, W1) split the output dimension
- Row-parallel layers (W_o, W2) split the input dimension and **ALL-REDUCE** the output
- LayerNorm, Dropout, Residuals operate on **full (B, S, h)** tensors

```
Flow: LayerNorm(B,S,h) -> QKV(B,S,h/N) -> Attention -> W_o -> ALL-REDUCE -> (B,S,h)
            full              TP region            TP region              full
```

In [ ]:
import torch
n_gpus = torch.cuda.device_count()
print(f"Running Vanilla TP with {n_gpus} GPUs...")
print()
!torchrun --nproc_per_node={n_gpus} vanilla_tp.py

## Part 4: Run TP + Sequence Parallelism

In TP + SP:
- **SP Region**: LayerNorm, Dropout, Residuals operate on **(B, S/N, h)** - sequence sharded!
- **TP Region**: Attention and FFN matmuls operate on **(B, S, h/N)** - hidden sharded
- **Transitions**:
  - SP to TP: **ALL-GATHER** reconstructs full sequence
  - TP to SP: **REDUCE-SCATTER** sums partials AND scatters along sequence

```
Flow: LayerNorm(B,S/N,h) -> ALL-GATHER -> (B,S,h) -> QKV -> Attn -> W_o -> REDUCE-SCATTER -> (B,S/N,h)
           SP region          expand        full      TP region           compress         SP region
```

In [ ]:
import torch
n_gpus = torch.cuda.device_count()
print(f"Running TP + SP with {n_gpus} GPUs...")
print()
!torchrun --nproc_per_node={n_gpus} tp_sp.py

## Part 5: Side-by-Side Comparison

Let's compare the results from both runs.

In [ ]:
import json

with open('results_vanilla_tp.json') as f:
    vanilla = json.load(f)
with open('results_tp_sp.json') as f:
    tp_sp = json.load(f)

ws = vanilla['world_size']

print()
print("=" * 80)
print("  SEQUENCE PARALLELISM: SIDE-BY-SIDE COMPARISON")
print("=" * 80)
print(f"  {'Metric':<40} {'Vanilla TP':>18} {'TP + SP':>18}")
print("  " + "-" * 76)

print(f"  {'Parameters per GPU':<40} {vanilla['params_per_gpu']:>15,} {tp_sp['params_per_gpu']:>15,}")

print("  " + "-" * 76)
print(f"  {'Model memory (MB)':<40} {vanilla['mem_model_mb']:>18.1f} {tp_sp['mem_model_mb']:>18.1f}")

vm = vanilla['mem_peak_mb']
sm = tp_sp['mem_peak_mb']
savings = (vm - sm) / vm * 100 if vm > sm else 0
print(f"  {'Peak memory (MB)':<40} {vm:>18.1f} {sm:>18.1f}")
print(f"  {'  -> Memory savings':<40} {'---':>18} {f'{savings:.1f}% less' if savings > 0 else 'similar':>18}")

print("  " + "-" * 76)
print(f"  {'Forward time (ms)':<40} {vanilla['fwd_ms']:>18.2f} {tp_sp['fwd_ms']:>18.2f}")
print(f"  {'Backward time (ms)':<40} {vanilla['bwd_ms']:>18.2f} {tp_sp['bwd_ms']:>18.2f}")
print(f"  {'Full step time (ms)':<40} {vanilla['step_ms']:>18.2f} {tp_sp['step_ms']:>18.2f}")

print(f"  {'Throughput (tok/s)':<40} {vanilla['throughput']:>18,.0f} {tp_sp['throughput']:>18,.0f}")
speedup = vanilla['step_ms'] / tp_sp['step_ms'] if tp_sp['step_ms'] > 0 else 1.0
print(f"  {'  -> Relative speed':<40} {'1.00x':>18} {f'{speedup:.2f}x':>18}")

print("  " + "-" * 76)
print(f"  {'Loss (should match)':<40} {vanilla['loss']:>18.4f} {tp_sp['loss']:>18.4f}")
print("=" * 80)

## Part 6: Activation Shape Trace

Let's visualize exactly where memory is saved by looking at tensor shapes at each point in the network.

In [ ]:
import torch
n_gpus = torch.cuda.device_count()
!torchrun --nproc_per_node={n_gpus} trace_shapes.py

## Part 7: Communication Analysis

Verify that TP+SP uses the same communication volume as vanilla TP.

In [ ]:
import torch
n_gpus = torch.cuda.device_count()
!torchrun --nproc_per_node={n_gpus} analyze_comm.py

## Part 8: Visualizations

Let's create charts from the benchmark results to see the SP advantage at a glance.

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

with open('results_vanilla_tp.json') as f:
    vanilla = json.load(f)
with open('results_tp_sp.json') as f:
    tp_sp = json.load(f)

# ── Style setup ──
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#21262d',
    'font.family': 'monospace',
    'font.size': 11,
})

C_VANILLA = '#f97583'   # red
C_SP      = '#79c0ff'   # blue
C_ACCENT  = '#56d364'   # green for savings

# ── Figure 1: Peak Memory ──
fig, ax = plt.subplots(figsize=(8, 4.5))

labels = ['Model Memory', 'Peak Memory']
v_vals = [vanilla['mem_model_mb'], vanilla['mem_peak_mb']]
s_vals = [tp_sp['mem_model_mb'], tp_sp['mem_peak_mb']]

x = np.arange(len(labels))
w = 0.32

bars_v = ax.bar(x - w/2, v_vals, w, label='Vanilla TP', color=C_VANILLA, edgecolor='none', zorder=3)
bars_s = ax.bar(x + w/2, s_vals, w, label='TP + SP', color=C_SP, edgecolor='none', zorder=3)

# Annotate bars with values
for bar in bars_v:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
            f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=10, color=C_VANILLA)
for bar in bars_s:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
            f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=10, color=C_SP)

# Savings annotation on peak memory
savings_pct = (vanilla['mem_peak_mb'] - tp_sp['mem_peak_mb']) / vanilla['mem_peak_mb'] * 100
if savings_pct > 0:
    mid_y = (vanilla['mem_peak_mb'] + tp_sp['mem_peak_mb']) / 2
    ax.annotate(f'{savings_pct:.1f}% less',
                xy=(1 + w/2 + 0.03, tp_sp['mem_peak_mb']),
                xytext=(1 + w/2 + 0.35, mid_y),
                fontsize=12, fontweight='bold', color=C_ACCENT,
                arrowprops=dict(arrowstyle='->', color=C_ACCENT, lw=1.5),
                ha='left', va='center')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylabel('Memory (MB)', fontsize=12)
ax.set_title('GPU Memory: Vanilla TP vs TP + Sequence Parallelism', fontsize=14, pad=12)
ax.legend(loc='upper left', framealpha=0.3, fontsize=11)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('viz_memory_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Peak memory savings from SP: {savings_pct:.1f}%')

In [ ]:
# ── Figure 2: Timing Breakdown (Forward / Backward) ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# --- Left: stacked bar for fwd/bwd ---
ax = axes[0]
methods = ['Vanilla TP', 'TP + SP']
fwd = [vanilla['fwd_ms'], tp_sp['fwd_ms']]
bwd = [vanilla['bwd_ms'], tp_sp['bwd_ms']]

x = np.arange(len(methods))
w = 0.45
ax.bar(x, fwd, w, label='Forward', color=C_SP, edgecolor='none', zorder=3)
ax.bar(x, bwd, w, bottom=fwd, label='Backward', color='#d2a8ff', edgecolor='none', zorder=3)

for i in range(len(methods)):
    total = fwd[i] + bwd[i]
    ax.text(i, total + 0.3, f'{total:.1f} ms', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color='#c9d1d9')

ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=12)
ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title('Step Time Breakdown', fontsize=13, pad=10)
ax.legend(framealpha=0.3, fontsize=10)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# --- Right: throughput ---
ax = axes[1]
tput = [vanilla['throughput'], tp_sp['throughput']]
colors = [C_VANILLA, C_SP]
bars = ax.bar(x, tput, w, color=colors, edgecolor='none', zorder=3)
for i, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(tput)*0.02,
            f'{tput[i]:,.0f}', ha='center', va='bottom', fontsize=11,
            fontweight='bold', color=colors[i])

ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=12)
ax.set_ylabel('Tokens / second', fontsize=12)
ax.set_title('Training Throughput', fontsize=13, pad=10)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('viz_timing.png', dpi=150, bbox_inches='tight')
plt.show()

speedup = vanilla['step_ms'] / tp_sp['step_ms'] if tp_sp['step_ms'] > 0 else 1.0
print(f'Speedup: {speedup:.2f}x  (SP should be ≈1.0x — same comm volume, just reorganized)')

### Theoretical Activation Memory Breakdown

This chart shows where SP saves memory — the components in the SP region (LayerNorm, Dropout, Residuals) shrink by 1/N, while the TP-region components (QKV, Attention, FFN) stay the same.

In [ ]:
# ── Figure 3: Per-Component Activation Memory Breakdown ──
# Theoretical activation sizes per layer per GPU (in elements, batch=B, seq=S, hidden=h, N=TP degree)

from config import BATCH_SIZE as B, SEQ_LEN as S, D_MODEL as h, D_FF as d_ff, N_HEADS as H
N = vanilla['world_size']

components = [
    'LayerNorm\n(×2)',
    'Residual\nsaved (×2)',
    'Dropout',
    'QKV\nprojections',
    'Attention\nscores',
    'FFN\nintermediate',
]

# Vanilla TP sizes (elements)
vanilla_sizes = np.array([
    2 * B * S * h,           # LayerNorm ×2
    2 * B * S * h,           # Residual saved ×2
    B * S * h,               # Dropout
    3 * B * S * (h // N),    # Q, K, V
    B * (H // N) * S * S,    # Attention scores
    B * S * (d_ff // N),     # FFN intermediate
], dtype=float)

# TP + SP sizes (elements)
sp_sizes = np.array([
    2 * B * (S // N) * h,        # LayerNorm ×2  ← SAVED
    2 * B * (S // N) * h,        # Residual ×2   ← SAVED
    B * (S // N) * h,            # Dropout        ← SAVED
    3 * B * S * (h // N),        # Q, K, V (same)
    B * (H // N) * S * S,        # Attention (same)
    B * S * (d_ff // N),         # FFN (same)
], dtype=float)

# Convert to MB (BF16 = 2 bytes)
vanilla_mb = vanilla_sizes * 2 / 1e6
sp_mb = sp_sizes * 2 / 1e6

fig, ax = plt.subplots(figsize=(11, 5))

x = np.arange(len(components))
w = 0.32

bars_v = ax.bar(x - w/2, vanilla_mb, w, label='Vanilla TP', color=C_VANILLA, edgecolor='none', zorder=3)
bars_s = ax.bar(x + w/2, sp_mb, w, label='TP + SP', color=C_SP, edgecolor='none', zorder=3)

# Mark the saved components
for i in range(3):  # first 3 components are in SP region
    saved = vanilla_mb[i] - sp_mb[i]
    if saved > 0:
        ax.text(i, max(vanilla_mb[i], sp_mb[i]) + max(vanilla_mb)*0.03,
                f'↓{saved/vanilla_mb[i]*100:.0f}%', ha='center', fontsize=10,
                fontweight='bold', color=C_ACCENT)

for i in range(3, len(components)):  # TP region components
    ax.text(i, max(vanilla_mb[i], sp_mb[i]) + max(vanilla_mb)*0.03,
            'same', ha='center', fontsize=9, color='#8b949e', style='italic')

# SP region / TP region labels
ymax = ax.get_ylim()[1]
ax.axvspan(-0.5, 2.5, alpha=0.07, color=C_SP, zorder=0)
ax.axvspan(2.5, len(components)-0.5, alpha=0.07, color=C_VANILLA, zorder=0)
ax.text(1.0, ymax * 0.95, 'SP Region\n(sequence sharded)', ha='center',
        fontsize=10, color=C_SP, fontweight='bold', va='top')
ax.text(4.0, ymax * 0.95, 'TP Region\n(hidden sharded)', ha='center',
        fontsize=10, color=C_VANILLA, fontweight='bold', va='top')

ax.set_xticks(x)
ax.set_xticklabels(components, fontsize=10)
ax.set_ylabel('Activation Memory per Layer (MB, BF16)', fontsize=11)
ax.set_title(f'Per-Component Activation Memory  (B={B}, S={S}, h={h}, TP={N})',
             fontsize=13, pad=12)
ax.legend(loc='upper right', framealpha=0.3, fontsize=11)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('viz_activation_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

total_v = vanilla_mb.sum()
total_s = sp_mb.sum()
print(f'Total activation memory per layer: Vanilla TP = {total_v:.1f} MB,  TP+SP = {total_s:.1f} MB')
print(f'Overall savings: {(total_v - total_s)/total_v*100:.1f}%')

In [ ]:
# ── Figure 4: Communication Pattern — Vanilla TP vs TP+SP ──
# Shows that the total bytes are identical; SP just splits the all-reduce

fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'wspace': 0.35})

comm_per_allreduce = B * S * h * 2  # bytes (BF16)
comm_mb = comm_per_allreduce / 1e6

# --- Left: Vanilla TP ---
ax = axes[0]
ops = ['All-Reduce\n(after W_o)', 'All-Reduce\n(after W2)']
vals = [comm_mb, comm_mb]
bars_l = ax.bar(ops, vals, color=C_VANILLA, edgecolor='none', width=0.5, zorder=3)
for bar in bars_l:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + comm_mb*0.03,
            f'{bar.get_height():.2f} MB', ha='center', fontsize=11,
            fontweight='bold', color=C_VANILLA)
ax.set_title('Vanilla TP\n(2 all-reduces per layer)', fontsize=12, color=C_VANILLA, pad=10)
ax.set_ylabel('Communication Volume (MB)', fontsize=11)
ax.set_ylim(0, comm_mb * 1.45)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

total_vanilla = sum(vals)
# Use axes transform with a sensible y in [0, 1] range
ax.text(0.5, 0.93, f'Total: {total_vanilla:.2f} MB',
        ha='center', fontsize=11, transform=ax.transAxes,
        color='#c9d1d9', style='italic')

# --- Right: TP + SP ---
ax = axes[1]
ops_sp = ['All-Gather\n(before QKV)', 'Reduce-Scatter\n(after W_o)',
          'All-Gather\n(before W1)', 'Reduce-Scatter\n(after W2)']
# Each AG or RS transfers (N-1)/N of the data, same as one "half" of all-reduce.
# Two (RS + AG) pairs = same total as two all-reduces.
vals_sp = [comm_mb/2, comm_mb/2, comm_mb/2, comm_mb/2]
colors_sp = [C_SP, '#d2a8ff', C_SP, '#d2a8ff']
bars_r = ax.bar(range(len(ops_sp)), vals_sp, color=colors_sp, edgecolor='none', width=0.55, zorder=3)
for i, bar in enumerate(bars_r):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + comm_mb*0.03,
            f'{vals_sp[i]:.2f}', ha='center', fontsize=10,
            fontweight='bold', color=colors_sp[i])
ax.set_xticks(range(len(ops_sp)))
ax.set_xticklabels(ops_sp, fontsize=9)
ax.set_title('TP + SP\n(2 reduce-scatters + 2 all-gathers)', fontsize=12, color=C_SP, pad=10)
ax.set_ylim(0, comm_mb * 1.45)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

total_sp = sum(vals_sp)
ax.text(0.5, 0.93, f'Total: {total_sp:.2f} MB  (identical!)',
        ha='center', fontsize=11, transform=ax.transAxes,
        color=C_ACCENT, fontweight='bold', style='italic')

# Bottom annotation
fig.text(0.5, 0.01, 'ALL-REDUCE  =  REDUCE-SCATTER + ALL-GATHER   \u2192   Same total bytes!',
         ha='center', fontsize=12, color=C_ACCENT, fontweight='bold')

plt.tight_layout(rect=[0, 0.05, 1, 1])  # leave room for bottom text
plt.savefig('viz_communication.png', dpi=150, bbox_inches='tight')
plt.show()


### Projected Memory Savings at Scale

How much does SP save for a real-world model (7B params) as we increase the TP degree?

In [ ]:
# ── Figure 5: Projected savings at different TP degrees (7B-scale model) ──
#
# 7B-class params: h=4096, d_ff=11008, H=32, S=2048, B=8

h_7b, dff_7b, H_7b, S_7b, B_7b, n_layers_7b = 4096, 11008, 32, 2048, 8, 32
tp_degrees = [1, 2, 4, 8]

vanilla_per_layer = []  # MB
sp_per_layer = []       # MB

for N_tp in tp_degrees:
    # Vanilla TP activation per layer (BF16, 2 bytes)
    ln  = 2 * B_7b * S_7b * h_7b                       # LayerNorm ×2
    res = 2 * B_7b * S_7b * h_7b                       # Residual ×2
    do  = B_7b * S_7b * h_7b                            # Dropout
    qkv = 3 * B_7b * S_7b * (h_7b // max(N_tp, 1))    # QKV
    att = B_7b * (H_7b // max(N_tp, 1)) * S_7b * S_7b # Attn scores
    ffn = B_7b * S_7b * (dff_7b // max(N_tp, 1))      # FFN
    vanilla_per_layer.append((ln + res + do + qkv + att + ffn) * 2 / 1e6)

    # TP + SP
    ln_sp  = 2 * B_7b * (S_7b // max(N_tp, 1)) * h_7b
    res_sp = 2 * B_7b * (S_7b // max(N_tp, 1)) * h_7b
    do_sp  = B_7b * (S_7b // max(N_tp, 1)) * h_7b
    sp_per_layer.append((ln_sp + res_sp + do_sp + qkv + att + ffn) * 2 / 1e6)

vanilla_total = [v * n_layers_7b for v in vanilla_per_layer]
sp_total = [s * n_layers_7b for s in sp_per_layer]
savings_gb = [(v - s) / 1e3 for v, s in zip(vanilla_total, sp_total)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: Per-layer activation memory ---
ax = axes[0]
x = np.arange(len(tp_degrees))
w = 0.32
ax.bar(x - w/2, vanilla_per_layer, w, label='Vanilla TP', color=C_VANILLA, edgecolor='none', zorder=3)
ax.bar(x + w/2, sp_per_layer, w, label='TP + SP', color=C_SP, edgecolor='none', zorder=3)

for i in range(len(tp_degrees)):
    if tp_degrees[i] > 1:
        pct = (vanilla_per_layer[i] - sp_per_layer[i]) / vanilla_per_layer[i] * 100
        ax.text(i, max(vanilla_per_layer[i], sp_per_layer[i]) + 15,
                f'-{pct:.0f}%', ha='center', fontsize=10, fontweight='bold', color=C_ACCENT)

ax.set_xticks(x)
ax.set_xticklabels([f'TP={n}' for n in tp_degrees], fontsize=12)
ax.set_ylabel('Activation Memory per Layer (MB)', fontsize=11)
ax.set_title('7B Model: Per-Layer Activations', fontsize=13, pad=10)
ax.legend(framealpha=0.3, fontsize=10)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# --- Right: Total savings in GB ---
ax = axes[1]
bars = ax.bar(x, savings_gb, 0.5, color=C_ACCENT, edgecolor='none', zorder=3)
for i, bar in enumerate(bars):
    if savings_gb[i] > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{savings_gb[i]:.1f} GB', ha='center', fontsize=11,
                fontweight='bold', color=C_ACCENT)

ax.set_xticks(x)
ax.set_xticklabels([f'TP={n}' for n in tp_degrees], fontsize=12)
ax.set_ylabel('Memory Saved (GB)', fontsize=11)
ax.set_title('7B Model: Total Activation Savings from SP\n(32 layers, B=8, S=2048)', fontsize=13, pad=10)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('viz_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

print('Note: TP=1 has no sharding at all, so SP has nothing to split (savings = 0).')
print(f'At TP=8, SP saves ~{savings_gb[3]:.1f} GB of activation memory per GPU — often the difference')
print(f'between fitting and not fitting a training run.')

In [ ]:
# ── Figure 6: Activation Shape Journey Through One Transformer Block ──
# A "waterfall" chart showing how the tensor size changes at each step

N = vanilla['world_size']

steps = [
    'Block\nInput',
    'Layer-\nNorm 1',
    'All-\nGather',
    'QKV\nProj',
    'Attn\nScores',
    'Attn\nOut',
    'W_o\nPartial',
    'Reduce-\nScatter',
    'Resid\nAdd',
    'Layer-\nNorm 2',
    'All-\nGather',
    'W1\n(FFN)',
    'GeLU',
    'W2\nPartial',
    'Reduce-\nScatter',
    'Dropout',
    'Block\nOutput',
]

# Vanilla TP element counts per GPU (for batch item 0)
vanilla_elems = [
    S * h,          # input
    S * h,          # LN1
    S * h,          # (no all-gather in vanilla, stays full)
    S * (h//N) * 3, # QKV
    (H//N)*S*S,     # attn scores
    S * (h//N),     # attn out
    S * h,          # W_o partial (before all-reduce)
    S * h,          # after all-reduce
    S * h,          # residual
    S * h,          # LN2
    S * h,          # (no all-gather)
    S * (d_ff//N),  # W1
    S * (d_ff//N),  # GeLU
    S * h,          # W2 partial
    S * h,          # after all-reduce
    S * h,          # dropout
    S * h,          # output
]

# TP+SP element counts per GPU
sp_elems = [
    (S//N) * h,     # input (SP)
    (S//N) * h,     # LN1 (SP)
    S * h,          # after all-gather
    S * (h//N) * 3, # QKV
    (H//N)*S*S,     # attn scores
    S * (h//N),     # attn out
    S * h,          # W_o partial
    (S//N) * h,     # after reduce-scatter (SP!)
    (S//N) * h,     # residual (SP)
    (S//N) * h,     # LN2 (SP)
    S * h,          # after all-gather
    S * (d_ff//N),  # W1
    S * (d_ff//N),  # GeLU
    S * h,          # W2 partial
    (S//N) * h,     # after reduce-scatter (SP!)
    (S//N) * h,     # dropout (SP)
    (S//N) * h,     # output (SP)
]

fig, ax = plt.subplots(figsize=(15, 5))

x = np.arange(len(steps))
ax.plot(x, vanilla_elems, 'o-', color=C_VANILLA, linewidth=2.5, markersize=7,
        label='Vanilla TP', zorder=4)
ax.plot(x, sp_elems, 's-', color=C_SP, linewidth=2.5, markersize=7,
        label='TP + SP', zorder=4)

# Shade SP vs TP regions
sp_ranges = [(0, 1.5), (7, 10.5), (14, 16.5)]  # SP region boundaries
for lo, hi in sp_ranges:
    ax.axvspan(lo - 0.3, hi + 0.3, alpha=0.08, color=C_SP, zorder=0)

# Mark comm operations
comm_steps = [2, 7, 10, 14]  # all-gather or reduce-scatter
for cs in comm_steps:
    ax.axvline(cs, color='#ffa657', alpha=0.4, linestyle='--', linewidth=1, zorder=1)

# Fill the gap between the two lines where SP saves
ax.fill_between(x, vanilla_elems, sp_elems,
                where=[v > s for v, s in zip(vanilla_elems, sp_elems)],
                alpha=0.15, color=C_ACCENT, zorder=2, label='Memory saved by SP')

ax.set_xticks(x)
ax.set_xticklabels(steps, fontsize=8.5, rotation=0)
ax.set_ylabel('Elements per GPU (batch=1)', fontsize=11)
ax.set_title('Activation Size at Each Step Through a Transformer Block', fontsize=14, pad=12)
ax.legend(loc='upper right', framealpha=0.3, fontsize=10)
ax.grid(axis='y', alpha=0.2, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add region annotations
ymax = ax.get_ylim()[1]
ax.text(0.5, ymax * 0.92, 'SP', fontsize=9, color=C_SP, ha='center', fontweight='bold')
ax.text(5.0, ymax * 0.92, 'TP Region', fontsize=9, color='#8b949e', ha='center')
ax.text(8.5, ymax * 0.92, 'SP', fontsize=9, color=C_SP, ha='center', fontweight='bold')
ax.text(12.0, ymax * 0.92, 'TP Region', fontsize=9, color='#8b949e', ha='center')
ax.text(15.5, ymax * 0.92, 'SP', fontsize=9, color=C_SP, ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('viz_shape_journey.png', dpi=150, bbox_inches='tight')
plt.show()

print('Green shaded area = memory saved by Sequence Parallelism')
print('Orange dashed lines = communication ops (all-gather / reduce-scatter)')
print('Blue shaded bands = SP region (sequence sharded, hidden full)')

In [ ]:
# ── Figure 7: Where Does Activation Memory Go? (Pie Charts) ──

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels_pie = ['LayerNorm\n(×2)', 'Residuals\n(×2)', 'Dropout', 'QKV', 'Attn Scores', 'FFN']
colors_pie = [C_SP, '#58a6ff', '#388bfd', C_VANILLA, '#f97583', '#d2a8ff']

# Vanilla TP
ax = axes[0]
wedges, texts, autotexts = ax.pie(
    vanilla_mb, labels=labels_pie, autopct='%1.0f%%',
    colors=colors_pie, startangle=90, pctdistance=0.75,
    textprops={'color': '#c9d1d9', 'fontsize': 9})
for at in autotexts:
    at.set_fontsize(8)
    at.set_color('#0d1117')
    at.set_fontweight('bold')
ax.set_title(f'Vanilla TP\n({vanilla_mb.sum():.0f} MB total)', fontsize=13, color=C_VANILLA, pad=10)

# TP + SP
ax = axes[1]
wedges, texts, autotexts = ax.pie(
    sp_mb, labels=labels_pie, autopct='%1.0f%%',
    colors=colors_pie, startangle=90, pctdistance=0.75,
    textprops={'color': '#c9d1d9', 'fontsize': 9})
for at in autotexts:
    at.set_fontsize(8)
    at.set_color('#0d1117')
    at.set_fontweight('bold')
ax.set_title(f'TP + SP\n({sp_mb.sum():.0f} MB total)', fontsize=13, color=C_SP, pad=10)

fig.suptitle('Activation Memory Distribution per Layer', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('viz_pie_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

print('The first three slices (LayerNorm, Residuals, Dropout) shrink by 1/N with SP.')
print('This shifts the bottleneck toward attention scores and FFN — addressable via')
print('Context Parallelism and activation checkpointing, respectively.')

## Part 9: Summary and Key Takeaways

### Memory Savings

| Component | Vanilla TP | TP + SP | Savings |
|-----------|-----------|---------|---------|
| LayerNorm | (B, S, h) | (B, S/N, h) | 1/N |
| Dropout | (B, S, h) | (B, S/N, h) | 1/N |
| Residuals | (B, S, h) | (B, S/N, h) | 1/N |
| Q, K, V | (B, S, h/N) | (B, S, h/N) | same |
| Attention | (B, H/N, S, S) | (B, H/N, S, S) | same |
| FFN | (B, S, d_ff/N) | (B, S, d_ff/N) | same |

### Communication Pattern

```
Vanilla TP:      2 ALL-REDUCES per layer
TP + SP:         2 REDUCE-SCATTERS + 2 ALL-GATHERS per layer

BUT: ALL-REDUCE = REDUCE-SCATTER + ALL-GATHER internally!

So: SAME total communication volume, just reorganized.
```

### When to Use SP

**Always use SP when using TP.** There is essentially no downside:
- Same communication volume
- Same (or very similar) throughput  
- 30-50% less activation memory
- Enables larger batch sizes or longer sequences

### What SP Doesn't Help

1. **Attention scores**: O(S^2) memory - need **Context Parallelism** for long sequences
2. **Model too large for TP=8**: Need **Pipeline Parallelism** to avoid slow inter-node TP

### The Parallelism Hierarchy

```
TP + SP (within node, NVLink)
    |
    v
Pipeline Parallelism (across nodes, for huge models)
    |
    v  
Data Parallelism + ZeRO (across pipeline replicas, for throughput)
    |
    v
Context Parallelism (orthogonal, for very long sequences)
```

In [ ]:
print('Tutorial complete!')